In [10]:
import os
from dotenv import load_dotenv
import polars as pl
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import Tool, AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

df = pl.read_csv(r"F:\Datasets\CSV datasets\sales.csv")

def query_dataframe(query: str) -> str:
    """Execute a Polars query on the dataframe"""
    try:
        namespace = {
            "df": df,
            "pl": pl,
            "__builtins__": {"len": len, "str": str, "int": int, "float": float, "print": print}
        }

        result = eval(query, namespace)

        if isinstance(result, pl.DataFrame):
            return f"Result DataFrame (showing first 10 rows):\n{result.head(10)}\nShape: {result.shape}"
        elif isinstance(result, pl.Series):
            return f"Result Series:\n{result.head(10)}\nLength: {len(result)}"
        else:
            return f"Result: {result}"

    except Exception as e:
        return f"Error executing query: {str(e)}\nMake sure to use valid Polars syntax."

polars_tool = Tool(
    name="polars_query",
    func=query_dataframe,
    description="""Use this to query data using Polars.
    The dataframe is available as 'df'.
    Example queries:
    - df.head()
    - df.describe()
    - df.columns
    - df.select(['column1', 'column2'])
    - df.filter(pl.col('price') > 100)
    - df.group_by('category').agg(pl.col('price').mean())
    - df.select(pl.col('column').value_counts())
    """
)

prompt = PromptTemplate.from_template("""You are a data analysis assistant with access to a Polars DataFrame.

Available columns: {columns}
Number of rows: {num_rows}
Data types: {dtypes}

You have access to the following tool:
{tools}

Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action (the Polars query to execute)
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Important: When using Action Input, provide only the Polars query code, nothing else.

Question: {input}
{agent_scratchpad}""")

tools = [polars_tool]
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)

context = {
    "columns": ", ".join(df.columns),
    "num_rows": len(df),
    "dtypes": ", ".join([f"{col}: {dtype}" for col, dtype in zip(df.columns, df.dtypes)])
}

def ask_question(question: str):
    """Ask a question about your data"""
    try:
        result = agent_executor.invoke({
            "input": question,
            **context
        })
        return result["output"]
    except Exception as e:
        return f"Error: {str(e)}"

def chat_with_data():
    """Interactive chat with your data"""
    print("Data Analysis Assistant Ready!")
    print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
    print(f"Columns: {', '.join(df.columns)}")
    print("\nAsk questions about your data (type 'exit' to quit):")

    while True:
        question = input("\nYou: ")
        if question.lower() in ['exit', 'quit', 'bye']:
            break

        answer = ask_question(question)
        print(f"\nAssistant: {answer}")

if __name__ == "__main__":
    print("=== Running Example Questions ===")

    questions = [
        "What are the columns in this dataset?",
        "Show me the first 5 rows of data",
        "What is the data type of each column?",
        "Calculate the total sales amount",
        "How many unique products are there?",
        "What's the average sale by product?"
    ]

    for question in questions:
        print(f"\nQ: {question}")
        answer = ask_question(question)
        print(f"A: {answer}")
        print("-" * 80)

    # chat_with_data()

=== Running Example Questions ===

Q: What are the columns in this dataset?


> Entering new AgentExecutor chain...
Thought: The question asks for the column names of the dataframe.
Action: polars_query
Action Input: df.columnsResult: ['user_id', 'age', 'sex', 'phone_number', 'joined_date', 'country', 'payment_method', 'loyalty_program_member', 'loyalty_points_redeemed', 'loyalty_tier', 'tier_discount_percentage', 'card_discount_percentage', 'coupon_discount_percentage', 'total_discount_percentage', 'total_purchase', 'total_discount', 'total_purchase_after_discount', 'transaction_id', 'payment_status', 'payment_date', 'payment_time', 'purchased_date', 'purchased_time', 'product_category', 'purchase_medium', 'return_status', 'refund_amount', 'return_date', 'order_id', 'released_date', 'estimated_delivery_date', 'received_date', 'total_delivery_days', 'shipping_method', 'shipping_cost', 'tracking_number', 'customer_exp_rating']I have successfully retrieved the column names.
Final Answer:

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash-exp"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 54
}
].


Thought: I need to find the number of unique product categories.
Action: polars_query
Action Input: df.select(pl.col('product_category').n_unique())Result DataFrame (showing first 10 rows):
shape: (1, 1)
┌──────────────────┐
│ product_category │
│ ---              │
│ u32              │
╞══════════════════╡
│ 20               │
└──────────────────┘
Shape: (1, 1)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash-exp"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 52
}
].


A: Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash-exp"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 52
}
]
--------------------------------------------------------------------------------

Q: What's the average sale by product?


> Entering new AgentExecutor chain...
A: Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-